##### Library Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1782506000.028970  779717 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


##### Loading Dataset (Single Participant)

In [2]:
df = pd.read_csv("functional_p18-s15.csv")

# print(df.head())
print(df.shape)

(85, 92)


##### PREPROCESSING

In [3]:
# Remove unnecessary columns
df = df.drop(columns=["SlNo","start_time", "end_time"])

# Seperate Features and Labels
X = df.drop(columns=['label'])
y = df['label']

# label encoding
encoder = LabelEncoder()
y = encoder.fit_transform(y)
# print(encoder.classes_)

# Train / Validation / Test Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2, # 20% Testing + Validation
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5, # 10% Testing, 10% Validation
    random_state=42,
    stratify=y_temp
)

# Feature Standardization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

##### NEURAL NETWORK

In [4]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),

    # Hidden Layer 1
    Dense(128, activation='relu'),
    Dropout(0.3),

    # Hidden Layer 2
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output Layer 
    Dense(1, activation='sigmoid')
])



W0000 00:00:1782506001.760081  779866 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1782506001.813213  779866 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1782506001.881862  779717 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [5]:
# Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
#View the model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        11,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,713 (77.00 KB)

 Trainable params: 19,713 (77.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

##### MODEL TRAINING

In [8]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1       
)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.5441 - loss: 0.7330 - val_accuracy: 0.3750 - val_loss: 0.7655
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5882 - loss: 0.7038 - val_accuracy: 0.2500 - val_loss: 0.7403
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6912 - loss: 0.6354 - val_accuracy: 0.3750 - val_loss: 0.7403
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7353 - loss: 0.5571 - val_accuracy: 0.3750 - val_loss: 0.7447
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7059 - loss: 0.5003 - val_accuracy: 0.3750 - val_loss: 0.7493
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7941 - loss: 0.5274 - val_accuracy: 0.3750 - val_loss: 0.7573
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7059 - loss: 0.5267 - val_accuracy: 0.3750 - val_loss: 0.7703
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7941 - loss: 0.4384 - val_accuracy: 0.3750 - val_loss:

##### EVALUATION ON TEST SET

In [9]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0   
)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.7436
Test Accuracy: 0.6667


In [10]:
probabilities = model.predict(X_test)
predictions = (probabilities > 0.5).astype(int)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


##### PERFORMANCE METRICS

In [11]:
print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall   :", recall_score(y_test, predictions))
print("F1 Score :", f1_score(y_test, predictions))

Accuracy : 0.6666666666666666
Precision: 0.6
Recall   : 0.75
F1 Score : 0.6666666666666666


In [12]:
# Confusion MAtrix
cm = confusion_matrix(y_test, predictions)
print(cm)

[[3 2]
 [1 3]]


In [13]:
# Classification Report
print(classification_report(
    y_test,
    predictions,
    target_names=encoder.classes_
))

              precision    recall  f1-score   support

  disengaged       0.75      0.60      0.67         5
     engaged       0.60      0.75      0.67         4

    accuracy                           0.67         9
   macro avg       0.68      0.68      0.67         9
weighted avg       0.68      0.67      0.67         9



##### SAVING DATA

In [14]:
# Save Model
model.save("neural-network.keras")

In [1]:
# Save Scaler
import joblib
joblib.dump.(scaler,"scaler.pkl")